# Classificador 3 — Lesão de Baixo Grau vs Alto Grau
Este notebook realiza o treinamento de um classificador para distinguir entre lesões de baixo e alto grau utilizando PyTorch com K-Fold cross-validation.

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Executando no dispositivo: {device}")

BASE_DIR = os.path.abspath(os.path.join(os.path.dirname("__file__"), ".."))
DATASET_PATH = os.path.join(BASE_DIR, "data/crops/c3_alto_vs_baixo")

IMG_SIZE = 70
BATCH_SIZE = 32
EPOCHS = 10
KFOLDS = 3

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

full_dataset = datasets.ImageFolder(DATASET_PATH, transform=transform)
class_names = full_dataset.classes
labels = [sample[1] for sample in full_dataset]

print(f"Número total de imagens: {len(full_dataset)}")
print(f"Classes: {class_names}")


In [ ]:
# K-Fold
kfold = StratifiedKFold(n_splits=KFOLDS, shuffle=True, random_state=42)

all_accuracies = []
all_precisions = []
all_recalls = []
all_f1s = []
all_specificities = []

for fold_idx, (train_idx, val_idx) in enumerate(kfold.split(np.zeros(len(labels)), labels)):
    print(f"\n=== Treinando Fold {fold_idx+1} ===")

    train_subset = Subset(full_dataset, train_idx)
    val_subset = Subset(full_dataset, val_idx)

    train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

    print(f"Treinamento com {len(train_subset)} imagens / Validação com {len(val_subset)} imagens")

    model = models.efficientnet_b0(weights='IMAGENET1K_V1')
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, 1)
    model = model.to(device)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.0003)

    model.train()
    for epoch in range(EPOCHS):
        loop = tqdm(train_loader, desc=f"Epoch [{epoch+1}/{EPOCHS}]", leave=False)
        for inputs, labels_batch in loop:
            inputs = inputs.to(device)
            labels_batch = labels_batch.float().unsqueeze(1).to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels_batch)
            loss.backward()
            optimizer.step()
            loop.set_postfix(loss=loss.item())

    # Avaliação
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for inputs, labels_batch in val_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            probs = torch.sigmoid(outputs).cpu().numpy()
            preds = (probs > 0.5).astype(int).flatten()
            y_pred.extend(preds)
            y_true.extend(labels_batch.numpy())

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    spec = tn / (tn + fp)

    all_accuracies.append(acc)
    all_precisions.append(prec)
    all_recalls.append(rec)
    all_f1s.append(f1)
    all_specificities.append(spec)

    print(f"Fold {fold_idx+1} - Acurácia: {acc:.4f}, F1: {f1:.4f}")

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(4, 3))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
    plt.xlabel("Predito")
    plt.ylabel("Real")
    plt.title(f"Matriz de Confusão - Fold {fold_idx+1}")
    plt.tight_layout()
    plt.savefig(f"../results/c3/matriz_confusao_fold_{fold_idx+1}.png")
    plt.close()



=== Treinando Fold 1 ===
Treinamento com 3170 imagens / Validação com 1585 imagens


Fold 1 - Acurácia: 0.9338, F1: 0.9178

=== Treinando Fold 2 ===
Treinamento com 3170 imagens / Validação com 1585 imagens


Fold 2 - Acurácia: 0.9300, F1: 0.9172

=== Treinando Fold 3 ===
Treinamento com 3170 imagens / Validação com 1585 imagens


Fold 3 - Acurácia: 0.9155, F1: 0.9003


In [ ]:
# Resultados finais
torch.save(model.state_dict(), '../models/classificador3.pth')
mean_acc = np.mean(all_accuracies)
mean_prec = np.mean(all_precisions)
mean_rec = np.mean(all_recalls)
mean_f1 = np.mean(all_f1s)
mean_spec = np.mean(all_specificities)

print("\n==== MÉDIAS FINAIS ====")
print(f"Acurácia média      : {mean_acc:.4f}")
print(f"Precisão média      : {mean_prec:.4f}")
print(f"Revocação média     : {mean_rec:.4f}")
print(f"F1-Score médio      : {mean_f1:.4f}")
print(f"Especificidade média: {mean_spec:.4f}")

# Salvar CSV
df_resultados = pd.DataFrame({
    "Fold": list(range(1, KFOLDS + 1)),
    "Acuracia": all_accuracies,
    "Precisao": all_precisions,
    "Revocacao": all_recalls,
    "F1": all_f1s,
    "Especificidade": all_specificities
})
df_resultados.loc["Media"] = df_resultados.mean(numeric_only=True)
df_resultados.to_csv("../results/c3/metricas_folds.csv", index=False)
print("Resultados salvos em resultados_classificador3.csv")



==== MÉDIAS FINAIS ====
Acurácia média      : 0.9264
Precisão média      : 0.9060
Revocação média     : 0.9186
F1-Score médio      : 0.9118
Especificidade média: 0.9319
Resultados salvos em resultados_classificador3.csv
